# From Scratch: Random Forests 🌲🌲🌲

Welcome to the second half of our Random Forest lesson! 

Earlier, you played with the interactive random forest demo. You probably noticed a few things when you wiggled those sliders:
1. Increasing the number of trees (`n_estimators`) smoothed out the decision boundary.
2. Each individual tree looked a bit blocky and weird, and they all disagreed slightly.
3. Limiting `max_depth` prevented the model from drawing crazy, highly-specific boxes around single outlier points.

Now, we are going to build exactly what you saw under the hood using purely NumPy. 

**An important note before we start:** Notice that we are *not* importing PyTorch, JAX, or TensorFlow. There is **no gradient descent** happening here. Trees don't learn by taking small steps down a loss landscape; they learn via "greedy splitting"—making the best possible cut at each step based on the data they currently see.

Let's import our tools and get started!

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_moons
from sklearn.tree import DecisionTreeClassifier
from collections import Counter

# Set seed for reproducibility
np.random.seed(42)

## 1. The Core Math: Gini Impurity

Before we can build a tree, we need to know how to split data. A good split separates classes cleanly. A bad split leaves the classes mixed up. 

We measure this "mixed-up-ness" using **Gini Impurity**. For a dataset with classes $c$, where $p_i$ is the probability of picking a data point of class $i$, the Gini impurity is:

$$Gini = 1 - \sum (p_i)^2$$

A pure node (all one class) has a Gini of 0. A perfectly mixed node (50/50 split of two classes) has a Gini of 0.5.

In [ ]:
def gini_impurity(y):
    """
    Calculates the Gini impurity of an array of labels.
    
    Args:
        y (np.array): 1D array of class labels (e.g., [0, 1, 1, 0, 0])
        
    Returns:
        float: The Gini impurity.
    """
    # TODO: Implement Gini impurity
    # Hint: np.bincount might be helpful for getting class counts!
    raise NotImplementedError("Implement Gini impurity!")

In [ ]:
# --- TEST CELL: Gini Impurity ---
# A pure array should have 0 impurity
assert gini_impurity(np.array([1, 1, 1, 1])) == 0.0, "Failed pure array test"
# A 50/50 split should have 0.5 impurity
assert gini_impurity(np.array([0, 0, 1, 1])) == 0.5, "Failed 50/50 test"
# A 3/1 split
assert np.isclose(gini_impurity(np.array([0, 0, 0, 1])), 0.375), "Failed 3/1 test"
print("✅ Gini impurity tests passed! Great job.")

## 2. Searching for the Best Split

To build a decision tree, we need to find the best feature and the best threshold to split our data on. 

We will split this into two steps so you aren't writing one giant function. First, let's write a helper that actually physically splits the data arrays given a feature index and a threshold.

In [ ]:
def split_dataset(X, y, feature_idx, threshold):
    """
    Splits X and y into two groups based on a threshold for a specific feature.
    Left group is <= threshold. Right group is > threshold.
    
    Args:
        X (np.array): 2D array of features
        y (np.array): 1D array of labels
        feature_idx (int): The column index in X to split on
        threshold (float): The value to split at
        
    Returns:
        tuple: (X_left, y_left, X_right, y_right)
    """
    # TODO: Create a boolean mask and use it to split X and y
    raise NotImplementedError("Implement split_dataset!")

In [ ]:
# --- TEST CELL: Split Dataset ---
X_test = np.array([[1, 2], [3, 4], [5, 6], [7, 8]])
y_test = np.array([0, 0, 1, 1])

X_l, y_l, X_r, y_r = split_dataset(X_test, y_test, feature_idx=0, threshold=4)
assert len(y_l) == 2 and len(y_r) == 2, "Split sizes are wrong"
assert np.all(y_l == 0) and np.all(y_r == 1), "Data routed to wrong sides"
print("✅ Dataset splitting passed!")

Now for the actual search. 

We need to iterate through **every feature** and **every unique value** in that feature. For each possible split, we calculate the weighted average of the Gini impurity of the left and right sides. We want to find the split that minimizes this weighted Gini.

*Note on `max_features`: Random Forests get their "randomness" by only looking at a random subset of features at each split. We'll build that logic in right now so it's ready later.*

In [ ]:
def find_best_split(X, y, max_features=None):
    """
    Finds the best feature and threshold to split on to minimize weighted Gini impurity.
    
    Args:
        X (np.array): 2D array of features
        y (np.array): 1D array of labels
        max_features (int): Number of random features to consider. If None, consider all.
        
    Returns:
        dict: A dictionary containing:
              {'feature': best_feature_idx, 
               'threshold': best_threshold, 
               'gini': best_weighted_gini}
              Returns None if no valid split is found.
    """
    n_samples, n_features = X.shape
    
    # Randomly select a subset of features if max_features is specified
    if max_features is not None:
        feature_indices = np.random.choice(n_features, max_features, replace=False)
    else:
        feature_indices = range(n_features)
        
    best_gini = float('inf')
    best_split = None
    
    # TODO: Loop over feature_indices
    # TODO: Inside, loop over every unique value in that feature column to use as a threshold
    # TODO: Split the dataset using your helper function
    # TODO: Calculate the weighted Gini impurity: 
    #       (len(left)/total) * gini(left) + (len(right)/total) * gini(right)
    # TODO: Keep track of the lowest Gini and its feature/threshold
    
    raise NotImplementedError("Implement find_best_split!")

In [ ]:
# --- TEST CELL: Find Best Split ---
X_test2 = np.array([[1], [2], [3], [4]])
y_test2 = np.array([0, 0, 1, 1])
split_info = find_best_split(X_test2, y_test2)

assert split_info is not None, "Failed: returned None"
assert split_info['feature'] == 0, "Wrong feature selected"
assert 2.0 <= split_info['threshold'] < 3.0, "Threshold should cleanly separate the 0s and 1s"
assert split_info['gini'] == 0.0, "Perfect split should have 0 weighted Gini"
print("✅ Best split logic passed!")

## 3. Building a Decision Tree

A decision tree is just a recursive application of `find_best_split`. We split the data, then we split the left side, then the right side, and so on, until we hit a stopping condition (like `max_depth` or a perfectly pure node).

To keep things organized, here is a simple `Node` class. It just stores information about the split, or the final prediction if it's a leaf.

In [ ]:
class Node:
    def __init__(self, feature=None, threshold=None, left=None, right=None, value=None):
        self.feature = feature      # Index of feature to split on
        self.threshold = threshold  # Value to split at
        self.left = left            # Left child Node
        self.right = right          # Right child Node
        self.value = value          # For leaf nodes: the predicted class
        
    def is_leaf(self):
        return self.value is not None

Now, implement the recursive builder and the predictor. Don't overthink the recursion! If you hit a stopping condition, return a `Node` with a `value`. Otherwise, find the best split, recursively call `build_tree` for left and right, and return a `Node` linking them.

In [ ]:
def build_tree(X, y, depth=0, max_depth=10, max_features=None):
    """
    Recursively builds the decision tree.
    """
    n_samples, n_features = X.shape
    n_classes = len(np.unique(y))
    
    # TODO: Stopping conditions
    # 1. If depth >= max_depth, or
    # 2. If n_classes == 1 (pure node):
    # -> Find the most common label in y and return a leaf Node (e.g., Node(value=most_common))
    
    # TODO: Find the best split
    # TODO: If no valid split is found (find_best_split returns None), return a leaf Node
    
    # TODO: Use split_dataset to actually divide X and y
    # TODO: Recursively call build_tree for left and right children
    # TODO: Return a Node containing the feature, threshold, left child, and right child
    
    raise NotImplementedError("Implement build_tree!")

def predict_single(node, x):
    """Walks down the tree to predict a single data point x."""
    # TODO: If node is a leaf, return its value
    # TODO: Otherwise, check if x[node.feature] <= node.threshold
    # TODO: Traverse left or right recursively
    raise NotImplementedError("Implement predict_single!")

def predict_tree(tree, X):
    """Predicts for an array of samples."""
    return np.array([predict_single(tree, x) for x in X])

In [ ]:
# --- TEST CELL: Single Tree vs Sklearn ---
# Generate some nonlinear toy data
X_train, y_train = make_moons(n_samples=150, noise=0.2, random_state=42)

# Train our tree
my_tree = build_tree(X_train, y_train, max_depth=3)
my_preds = predict_tree(my_tree, X_train)
my_acc = np.mean(my_preds == y_train)

# Train sklearn's tree
sk_tree = DecisionTreeClassifier(max_depth=3, random_state=42)
sk_tree.fit(X_train, y_train)
sk_preds = sk_tree.predict(X_train)
sk_acc = np.mean(sk_preds == y_train)

print(f"Your Tree Accuracy: {my_acc:.3f}")
print(f"Sklearn Tree Accuracy: {sk_acc:.3f}")
assert my_acc >= 0.85, "Accuracy seems a bit too low, check your split logic!"
print("✅ Single Tree is fully functional! Look at you go.")

## 4. The Magic of Bagging (Random Forest)

Remember how in the demo, every single tree in the forest got a slightly different, weirdly shaped decision boundary? And then we averaged them together to get a smooth, robust model?

That happens because of two things:
1. **Feature Subsampling:** (We already handled this with `max_features`).
2. **Bootstrapping:** Each tree is trained on a random sample of the data, drawn *with replacement*, of the exact same size as the original dataset.

Let's implement the bootstrap!

In [ ]:
def bootstrap_sample(X, y):
    """
    Creates a bootstrap sample of X and y (same size as original, but sampled with replacement).
    
    Args:
        X (np.array): Features
        y (np.array): Labels
        
    Returns:
        tuple: (X_boot, y_boot)
    """
    # TODO: Generate a list of random indices of length len(X) with replacement
    # TODO: Return X and y indexed by these random indices
    raise NotImplementedError("Implement bootstrap_sample!")

In [ ]:
# --- TEST CELL: Bootstrapping ---
X_boot_test = np.arange(5).reshape(-1, 1)
y_boot_test = np.arange(5)
X_b, y_b = bootstrap_sample(X_boot_test, y_boot_test)

assert len(X_b) == 5, "Sample should be the same length as original"
assert len(np.unique(X_b)) < 5, "You probably didn't use 'replacement=True'. All elements are unique."
print("✅ Bootstrapping works!")

Now, bring it all together into our final `RandomForest` class. 

For predictions, since we are doing classification, we will take all the predictions from our trees and use a **majority vote**.

In [ ]:
class RandomForest:
    def __init__(self, n_estimators=10, max_depth=10, max_features=None):
        self.n_estimators = n_estimators
        self.max_depth = max_depth
        self.max_features = max_features
        self.trees = []
        
    def fit(self, X, y):
        self.trees = []
        # TODO: Loop n_estimators times. 
        # TODO: In each loop, create a bootstrap sample.
        # TODO: Build a tree using that sample. Don't forget to pass max_depth and max_features!
        # TODO: Append the tree to self.trees
        raise NotImplementedError("Implement Random Forest fit!")
            
    def predict(self, X):
        # TODO: Get predictions for X from every tree in self.trees
        # TODO: For each data point in X, find the most common prediction (majority vote)
        # TODO: Return the final array of predictions
        raise NotImplementedError("Implement Random Forest predict!")

In [ ]:
# --- TEST CELL: Random Forest ---
rf = RandomForest(n_estimators=15, max_depth=5, max_features=1) 
rf.fit(X_train, y_train)
rf_preds = rf.predict(X_train)
rf_acc = np.mean(rf_preds == y_train)

print(f"Random Forest Accuracy: {rf_acc:.3f}")
assert len(rf.trees) == 15, "Did not store the correct number of trees"
assert rf_acc >= 0.85, "Accuracy is unexpectedly low"
print("✅ Random Forest successfully trained!")

## 5. Visualizing the Decision Boundary (Building Intuition)

We've built it. Now let's *see* it. We've provided a helper function below to plot the decision boundary of your models. 

Run the cell below to train a Single Tree and your Random Forest side-by-side, and plot them. Pay close attention to how much "smoother" the forest looks compared to the single tree, just like you saw when wiggling the sliders in the interactive demo!

In [ ]:
def plot_decision_boundary(model, X, y, title, ax):
    x_min, x_max = X[:, 0].min() - 0.5, X[:, 0].max() + 0.5
    y_min, y_max = X[:, 1].min() - 0.5, X[:, 1].max() + 0.5
    xx, yy = np.meshgrid(np.arange(x_min, x_max, 0.02),
                         np.arange(y_min, y_max, 0.02))
    
    # Sklearn uses .predict(), our trees use predict_tree
    if hasattr(model, 'predict'): 
        Z = model.predict(np.c_[xx.ravel(), yy.ravel()])
    else: 
        Z = predict_tree(model, np.c_[xx.ravel(), yy.ravel()])
        
    Z = Z.reshape(xx.shape)
    ax.contourf(xx, yy, Z, alpha=0.4, cmap="RdBu")
    ax.scatter(X[:, 0], X[:, 1], c=y, s=20, edgecolor='k', cmap="RdBu")
    ax.set_title(title)

# Generate a slightly harder dataset
X_viz, y_viz = make_moons(n_samples=200, noise=0.25, random_state=42)

# Train Single Tree (deep, prone to overfitting)
tree_viz = build_tree(X_viz, y_viz, max_depth=10)

# Train our Random Forest
rf_viz = RandomForest(n_estimators=30, max_depth=10, max_features=1)
rf_viz.fit(X_viz, y_viz)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
plot_decision_boundary(tree_viz, X_viz, y_viz, "Single Decision Tree", ax1)
plot_decision_boundary(rf_viz, X_viz, y_viz, "Our NumPy Random Forest", ax2)
plt.show()

## 6. Your Turn: Wiggle the "Sliders" in Code

In the interactive demo, you adjusted parameters and watched the model react. Now, do it in code. 

Try changing the variables in the cell below. 
*   What happens if you set `n_estimators=1`? 
*   What happens to the single tree if you change `max_depth` to `1` or `2`? 
*   What happens to the forest if you change `max_features` to `2` (meaning it considers all features at every split)? Does it look *more* or *less* like the single tree?

*Hint: If `max_features` is maxed out, the trees lose their diversity and bagging becomes much less effective!*

In [ ]:
# Tweak these hyperparameters!
MY_N_ESTIMATORS = 3   # Try 1, 3, 10, 50
MY_MAX_DEPTH = 15     # Try 1, 3, 15
MY_MAX_FEATURES = 1   # Try 1, 2 (since we only have 2 features in this dataset)

# Re-train and plot
experiment_rf = RandomForest(n_estimators=MY_N_ESTIMATORS, 
                             max_depth=MY_MAX_DEPTH, 
                             max_features=MY_MAX_FEATURES)
experiment_rf.fit(X_viz, y_viz)

fig, ax = plt.subplots(1, 1, figsize=(6, 5))
plot_decision_boundary(experiment_rf, X_viz, y_viz, 
                       f"RF (n={MY_N_ESTIMATORS}, depth={MY_MAX_DEPTH}, max_feat={MY_MAX_FEATURES})", ax)
plt.show()